# Validação de FIDC (`CotaFIDC`), Letras Financeiras (`LetraFinanceira`), Z-Spread ANBIMA e Carteira em Lote

Este notebook valida:
- **FIDC (`CotaFIDC`)**: ao menos 2 ativos por indexador (`DI + Spread`, `% do DI`, `IPCA`, `IGP-M`, `PRÉ`).
- **Letras Financeiras (`LetraFinanceira`)**: `SENIOR` (*yield to maturity*), `SUBORDINADA` / `COMPLEMENTAR` (*yield to call*), e consulta à matriz aberta de spreads indicativos da ANBIMA (`ANBIMADataClient.get_letras_financeiras()`).
- **Z-Spread ANBIMA**: exemplos oficiais do manual (CDI + spread, IPCA não incentivado, IPCA incentivado com *gross-up* de IR e `% do DI` com ETTJ).
- **Benchmark Cython (`price_credit_portfolio`)**: precificação em lote de carteira completa.

In [1]:
import datetime, os, time
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
import pandas as pd
from bz_pricing.data import ANBIMADataClient
from bz_pricing.instruments import CotaFIDC, LetraFinanceira, Debenture, price_credit_portfolio

ref_dt = datetime.date(2026, 7, 1)
anb_cli = ANBIMADataClient()

In [2]:
# 1. FIDC: 2 casos por indexador (DI+Spread, %DI, IPCA, IGP-M, PRÉ)
fidcs = [
    (CotaFIDC.from_schedule('FIDC-DI-1',   '2026-01-02', '2028-01-03', indexador='DI',    coupon_rate=0.020, coupon_freq='mensal'), 0.024, 1000.0, 1005.0, 0.135),
    (CotaFIDC.from_schedule('FIDC-DI-2',   '2025-07-01', '2029-07-02', indexador='DI',    coupon_rate=0.025, coupon_freq='semestral'), 0.028, 1000.0, 1020.0, 0.135),
    (CotaFIDC.from_schedule('FIDC-PCT-1',  '2026-01-02', '2028-01-03', indexador='DI',    coupon_rate=0.0, multiplicador_di=1.08, coupon_freq='mensal'), 1.10, 1000.0, 1004.0, 0.135),
    (CotaFIDC.from_schedule('FIDC-PCT-2',  '2025-07-01', '2028-07-03', indexador='DI',    coupon_rate=0.0, multiplicador_di=1.12, coupon_freq='semestral'), 1.15, 1000.0, 1030.0, 0.135),
    (CotaFIDC.from_schedule('FIDC-IPCA-1', '2025-01-15', '2028-01-17', indexador='IPCA',  coupon_rate=0.075, coupon_freq='semestral'), 0.080, 1100.0, None, None),
    (CotaFIDC.from_schedule('FIDC-IPCA-2', '2025-06-16', '2029-06-15', indexador='IPCA',  coupon_rate=0.082, coupon_freq='mensal'), 0.085, 1060.0, None, None),
    (CotaFIDC.from_schedule('FIDC-IGPM-1', '2025-01-15', '2028-01-17', indexador='IGP-M', coupon_rate=0.070, coupon_freq='semestral'), 0.075, 1090.0, None, None),
    (CotaFIDC.from_schedule('FIDC-IGPM-2', '2025-06-16', '2029-06-15', indexador='IGP-M', coupon_rate=0.078, coupon_freq='mensal'), 0.080, 1055.0, None, None),
    (CotaFIDC.from_schedule('FIDC-PRE-1',  '2026-01-02', '2028-01-03', indexador='PRÉ',   coupon_rate=0.145, coupon_freq='semestral'), 0.148, 1000.0, None, None),
    (CotaFIDC.from_schedule('FIDC-PRE-2',  '2025-07-01', '2029-07-02', indexador='PRÉ',   coupon_rate=0.150, coupon_freq='mensal'), 0.152, 1000.0, None, None),
]
# Ajustar flag is_pct_di nos dois casos %DI
fidcs[2][0].is_pct_di = True
fidcs[3][0].is_pct_di = True

fidc_rows = []
for obj, taxa, vna_v, pupar_v, di_c in fidcs:
    m = obj.risk_metrics(ref_dt, taxa, vna=vna_v, pu_par_val=pupar_v, di_curve=di_c)
    fidc_rows.append({'ticker': obj.ticker, 'indexador': obj.indexador, 'pct_di': obj._is_pct_di(), **m})
pd.DataFrame(fidc_rows)

,ticker,indexador,pct_di,pu_par,pu,cotacao,duration_du,duration_anos,modified_duration,dv01,convexity,z_spread
0,FIDC-DI-1,DI,False,1005.000000,999.711952,99.4738,339.61,1.347665,1.159541,0.115921,2.439479,0.024
1,FIDC-DI-2,DI,False,1020.000000,1012.730296,99.2872,616.47,2.446325,2.096646,0.212334,6.737424,0.028
2,FIDC-PCT-1,DI,True,1004.000000,1000.553103,99.6566,342.25,1.358140,1.181546,0.118220,2.517184,NaN
3,FIDC-PCT-2,DI,True,1030.000000,1023.141021,99.3340,443.23,1.758872,1.520517,0.155570,3.801520,NaN
4,FIDC-IPCA-1,IPCA,False,1136.257177,1128.707735,99.3355,361.95,1.436329,1.329934,0.150111,3.095056,NaN
5,FIDC-IPCA-2,IPCA,False,1063.652860,1055.972919,99.2779,659.35,2.616471,2.411494,0.254647,8.488502,NaN
6,FIDC-IGPM-1,IGP-M,False,1123.576267,1116.043152,99.3295,363.56,1.442731,1.342075,0.149781,3.139954,NaN
7,FIDC-IGPM-2,IGP-M,False,1058.464494,1053.314809,99.5134,662.94,2.630729,2.435860,0.256573,8.626266,NaN
8,FIDC-PRE-1,PRÉ,False,1067.749368,1064.082671,99.6565,331.22,1.314393,1.144942,0.121831,2.448169,NaN
9,FIDC-PRE-2,PRÉ,False,1000.000000,995.754860,99.5754,616.79,2.447593,2.124647,0.211563,6.953185,NaN


In [3]:
# 2. Letras Financeiras (Sênior YTM, Subordinada YTC e Matriz Aberta ANBIMA)
spread_abc_12m = LetraFinanceira.indicative_spread_from_anbima('ABC', 12, client=anb_cli)
spread_abc_24m = LetraFinanceira.indicative_spread_from_anbima('ABC', 24, client=anb_cli)
print(f'Spread indicativo ABC ANBIMA: 12M={spread_abc_12m*100:.4f}% | 24M={spread_abc_24m*100:.4f}%')

lfs = [
    (LetraFinanceira.from_bullet('LF-SEN-DI',    '2026-01-02', '2028-01-03', indexador='DI',   coupon_rate=spread_abc_24m, nominal_value=50000.0, classe='SENIOR', emissor='ABC'), spread_abc_24m, 50000.0, 51500.0, 0.135),
    (LetraFinanceira.from_bullet('LF-SEN-PRE',   '2026-01-02', '2028-01-03', indexador='PRÉ',  coupon_rate=0.140, nominal_value=50000.0, classe='SENIOR'), 0.142, 50000.0, None, None),
    (LetraFinanceira.from_bullet('LFSN-SUB-DI',  '2026-01-02', '2033-01-03', indexador='DI',   coupon_rate=0.012, nominal_value=50000.0, classe='SUBORDINADA', call_date='2031-01-02'), 0.014, 50000.0, 51500.0, 0.135),
    (LetraFinanceira.from_bullet('LFSC-COMP-IPCA','2025-01-15', '2032-01-15', indexador='IPCA', coupon_rate=0.072, nominal_value=50000.0, classe='COMPLEMENTAR', call_date='2030-01-15'), 0.075, 54000.0, None, None),
]
lf_rows = []
for obj, taxa, vna_v, pupar_v, di_c in lfs:
    m = obj.risk_metrics(ref_dt, taxa, vna=vna_v, pu_par_val=pupar_v, di_curve=di_c)
    lf_rows.append({'ticker': obj.ticker, 'classe': obj.classe, 'horizonte': obj.maturity_date, **m})
pd.DataFrame(lf_rows)

Spread indicativo ABC ANBIMA: 12M=0.2484% | 24M=0.3194%


,ticker,classe,horizonte,pu_par,pu,cotacao,duration_du,duration_anos,modified_duration,dv01,convexity,z_spread
0,LF-SEN-DI,SENIOR,2028-01-03,51500.000000,51500.000000,99.9999,378.0,1.500000,1.317378,6.784498,2.892476,0.003194
1,LF-SEN-PRE,SENIOR,2028-01-03,53274.474975,53134.585809,99.7374,378.0,1.500000,1.313485,6.979149,2.875405,NaN
2,LFSN-SUB-DI,SUBORDINADA,2031-01-02,51500.000000,51047.274070,99.1209,1127.0,4.472222,3.885882,19.836366,18.476490,0.014000
3,LFSC-COMP-IPCA,COMPLEMENTAR,2030-01-15,59721.164904,59138.562228,99.0244,884.0,3.507937,3.263197,19.298076,13.683985,NaN


In [4]:
# 3. Z-Spread ANBIMA (exemplos oficiais) + ETTJ ANBIMA + Carteira em Lote (Cython)
df_ettj = anb_cli.get_ettj()
print('ETTJ ANBIMA (primeiras linhas):')
display(df_ettj.head(5))

deb_ipca = Debenture.from_schedule('DEB-INC', '2025-01-15', '2030-01-15', indexador='IPCA', coupon_rate=0.065) if hasattr(Debenture, 'from_schedule') else CotaFIDC.from_schedule('DEB-INC', '2025-01-15', '2030-01-15', indexador='IPCA', coupon_rate=0.065)
m_nao_inc = deb_ipca.risk_metrics(ref_dt, 0.073348, vna=1080.0, risk_free_rate=0.064904, incentivada=False)
m_inc     = deb_ipca.risk_metrics(ref_dt, 0.073348, vna=1080.0, risk_free_rate=0.064904, incentivada=True)
print(f'Z-Spread Não Incentivada: {m_nao_inc["z_spread"]*100:.4f}% (manual: 0.7929%)')
print(f'Z-Spread Incentivada (gross-up): {m_inc["z_spread"]*100:.4f}% | Taxa gross-up: {m_inc["taxa_gross_up"]*100:.4f}%')

# Benchmark de carteira com 200 ativos via Cython
positions = [
    {'instrument': fidcs[0][0], 'discount_rate': 0.024, 'quantity': 50, 'vna': 1000.0, 'pu_par_val': 1005.0, 'di_curve': 0.135},
    {'instrument': deb_ipca,    'discount_rate': 0.073348, 'quantity': 100, 'vna': 1080.0, 'risk_free_rate': 0.064904, 'incentivada': True},
] * 100
t0 = time.perf_counter()
df_port = price_credit_portfolio(positions, ref_dt)
elapsed = time.perf_counter() - t0
print(f'Carteira de {len(df_port)} ativos precificada em {elapsed:.4f} s')
df_port.head(4)

ETTJ ANBIMA (primeiras linhas):


,du,ettj_ipca,ettj_pref,inflacao_implicita
0,126,0.042380,0.137883,0.091620
1,252,0.060467,0.141395,0.076313
2,378,0.069294,0.144730,0.070547
3,504,0.073602,0.147447,0.068782
4,630,0.075705,0.149541,0.068639


Z-Spread Não Incentivada: 0.7929% (manual: 0.7929%)
Z-Spread Incentivada (gross-up): 2.2092% | Taxa gross-up: 8.8430%


Carteira de 200 ativos precificada em 0.9879 s


,ticker,classe,indexador,quantity,pu_par,pu,valor_financeiro,cotacao,duration_du,modified_duration,dv01,dv01_financeiro,convexity,z_spread
0,FIDC-DI-1,CotaFIDC,DI,50.0,1005.000000,999.711952,49985.59,99.4738,339.61,1.159541,0.115921,5.796036,2.439479,0.024000
1,DEB-INC,CotaFIDC,IPCA,100.0,1083.243565,1056.980187,105698.01,97.5754,791.42,2.925954,0.309268,30.926758,11.932196,0.022092
2,FIDC-DI-1,CotaFIDC,DI,50.0,1005.000000,999.711952,49985.59,99.4738,339.61,1.159541,0.115921,5.796036,2.439479,0.024000
3,DEB-INC,CotaFIDC,IPCA,100.0,1083.243565,1056.980187,105698.01,97.5754,791.42,2.925954,0.309268,30.926758,11.932196,0.022092


## Referências Normativas e Bibliográficas

- **ANBIMA – Associação Brasileira das Entidades dos Mercados Financeiro e de Capitais.** *Metodologias ANBIMA de Precificação de FIDC*. Rio de Janeiro: ANBIMA. Arquivo local `dados/manuais_anbima/Metodologias_ANBIMA_de_Precificacao_de_FIDC_VF_5102094423.pdf`.
- **ANBIMA – Associação Brasileira das Entidades dos Mercados Financeiro e de Capitais.** *Metodologias de Precificação — Letras Financeiras*. Rio de Janeiro: ANBIMA. Arquivo local `dados/manuais_anbima/Metodologias_Precificacao_Letras_Financeiras_VF_c1c488f49d.pdf`.
- **ANBIMA – Associação Brasileira das Entidades dos Mercados Financeiro e de Capitais.** *Metodologia ANBIMA de Cálculo dos Spreads de Crédito (Z-Spread)*. Rio de Janeiro: ANBIMA. Arquivo local `dados/manuais_anbima/Metodologia_ANBIMA_de_calculo_dos_spreads_de_credito_42697e0db7.pdf`.
- **ANBIMA – Associação Brasileira das Entidades dos Mercados Financeiro e de Capitais.** *Dados Abertos ANBIMA — Preços e Taxas Indicativas (Letras Financeiras, FIDC e ETTJ)*. Disponível em: <https://data.anbima.com.br/>.